# LLM Evaluation: NLG Metrics, Semantic Similarity & LLM-as-a-Judge

## 1. Overview
Evaluating natural language generation (NLG) is one of the hardest challenges in artificial intelligence. Because there are infinite valid ways to answer an open-ended prompt, simple string matching fails.

In this notebook, we master the full spectrum of modern LLM evaluation:
1. **Statistical & N-gram Overlap Metrics**: Perplexity, BLEU, ROUGE, and METEOR.
2. **Model-Based Semantic Alignment**: BERTScore (contextualized token embeddings).
3. **LLM-as-a-Judge**: Pointwise scoring with rubrics and Pairwise comparison with position debiasing.
4. **Structured Schemas & Bias Mitigation**: Using Pydantic to extract reliable evaluation verdicts.

---
## 1. Statistical Overlap & Embedding Metrics (BLEU, ROUGE, Perplexity)

- **Perplexity**: The exponentiated cross-entropy loss; measures how surprised a language model is by text.
- **BLEU (Bilingual Evaluation Understudy)**: Precision-oriented n-gram overlap, standard for machine translation.
- **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**: Recall-oriented n-gram overlap, standard for summarization (ROUGE-1, ROUGE-2, ROUGE-L).
- **METEOR**: Incorporates stemming and WordNet synonym matching to overcome exact-match limitations.

In [9]:
#!pip install jiwer rouge-score

In [2]:
import numpy as np
from jiwer import wer
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction
from nltk.tokenize import word_tokenize

### Perplexity

Perplexity measures how well a language model predicts the next word in a sequence of words. Lower perplexity values mean the model is more confident and accurate in its predictions. 

The perplexity of a sequence of words \( W = (w_1, w_2, \dots, w_N) \) is given by:

$$
PP(W) = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log p(w_i | w_1, \dots, w_{i-1})\right)
$$

Where:
- $w_i$ is the $i$-th word in the sequence.
- $p(w_i | w_1, \dots, w_{i-1})$ is the probability of the word $w_i$ given the previous words (predicted by the model).
- $N$ is the total number of words in the sequence.

Alternatively, perplexity can also be expressed as:

$$
PP(W) = 2^{H(p)}
$$

Where:
- \( H(p) \) is the **cross-entropy** of the model, defined as the average number of bits required to represent each word.


In [3]:
def perplexity(probabilities):
    N = len(probabilities)
    log_prob_sum = np.sum(np.log2(probabilities))
    avg_log_prob = log_prob_sum / N
    perplexity = 2 ** (-avg_log_prob)
    return perplexity

# Example:

# Suppose you have a list of probabilities assigned by your language model to each word in a sentence
probabilities = [0.1, 0.2, 0.3, 0.4]  
perplexity_score = perplexity(probabilities)
print("Perplexity:", perplexity_score)

Perplexity: 4.518010018049224


### NLG Metrics

**BLEU (Bilingual Evaluation Understudy) Score:** 
- Compares n-grams of the generated text to reference n-grams, commonly used in machine translation.


$$
BLEU = BP \times \exp\left(\sum_{n=1}^{N} w_n \cdot \log p_n\right)
$$

Where:
- $BP$ is the **Brevity Penalty**, which corrects for shorter translations (if the hypothesis is shorter than the reference).
  
$$
BP = \begin{cases}
1 & \text{if } c > r \\
\exp(1 - \frac{r}{c}) & \text{if } c \leq r
\end{cases}
$$

Here:
- $c$ is the length of the generated (candidate) translation.
- $r$ is the length of the reference translation.
- $p_n$ is the **precision** of n-grams (n = 1, 2, 3,..., N).
  

**ROUGE Score:** 
- Evaluates automatic summarization by comparing n-grams, word sequences, or sentence-level overlap between the generated and reference texts.

- **ROUGE-1** measures the overlap of unigrams (individual words) between the generated text and the reference text.

- **ROUGE-2** measures the overlap of bigrams (pairs of adjacent words) between the generated text and the reference text.

- **ROUGE-L** measures the longest common subsequence (LCS) between the generated text and the reference text. The LCS is the longest sequence of words that appears in both texts in the same order (though not necessarily consecutively).


**WER (Word Error Rate) Score:** 
- Measures the difference between the generated and reference text by calculating the minimum number of insertions, deletions, and substitutions needed to match the two.

**CER (Character Error Rate) Score:** 
- Similar to WER, but calculates the error rate at the character level instead of the word level.WER and CER are specialized versions of the edit distance metric, focusing on words and characters, respectively, and are typically used in specific tasks like speech recognition and OCR.

In [4]:
# BLEU
def calculate_bleu(reference, hypothesis):
    smoothing = SmoothingFunction().method1
    return sentence_bleu([word_tokenize(reference)], word_tokenize(hypothesis), smoothing_function=smoothing)

# ROUGE
def calculate_rouge(reference, hypothesis):
    # Initialize scorer
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)    
    return scores
    
# Word Error Rate (WER)
def calculate_wer(reference, hypothesis):
    return wer(reference, hypothesis)

# Character Error Rate (CER)
def calculate_cer(reference, hypothesis):
    ref_chars = ''.join(reference.split())
    hyp_chars = ''.join(hypothesis.split())
    return wer(ref_chars, hyp_chars)

In [5]:
def computeMetrics (reference, hypothesis):
    # Calculate evaluation measures
    bleu = calculate_bleu(reference, hypothesis)
    rouge = calculate_rouge(reference, hypothesis)
    
    wer_score = calculate_wer(reference, hypothesis)
    cer = calculate_cer(reference, hypothesis)
        
    # Print results
    print("BLEU Score:", bleu)
    print("ROUGE Score:", rouge)
    print("WER Score:", wer_score)
    print("CER Score:", cer)

In [6]:
# Example 1
reference = "The quick brown fox jumps over the lazy dog"
hypothesis = "The quick brown fox jumps over the lazy dog"

computeMetrics(reference,hypothesis)

BLEU Score: 1.0
ROUGE Score: {'rouge1': Score(precision=1.0, recall=1.0, fmeasure=1.0), 'rouge2': Score(precision=1.0, recall=1.0, fmeasure=1.0), 'rougeL': Score(precision=1.0, recall=1.0, fmeasure=1.0)}
WER Score: 0.0
CER Score: 0.0


In [7]:
# Example 2
reference = "The quick brown fox jumps over the lazy dog"
hypothesis = "The quick brown fox jumps over the lazy cat"

computeMetrics(reference,hypothesis)

BLEU Score: 0.8633400213704505
ROUGE Score: {'rouge1': Score(precision=0.8888888888888888, recall=0.8888888888888888, fmeasure=0.8888888888888888), 'rouge2': Score(precision=0.875, recall=0.875, fmeasure=0.875), 'rougeL': Score(precision=0.8888888888888888, recall=0.8888888888888888, fmeasure=0.8888888888888888)}
WER Score: 0.1111111111111111
CER Score: 1.0


In [8]:
# Example 3
reference = "The quick brown fox jumps over the lazy dog"
hypothesis = "The fast brown fox jumps over the lazy cat"

computeMetrics(reference,hypothesis)

BLEU Score: 0.6104735835807844
ROUGE Score: {'rouge1': Score(precision=0.7777777777777778, recall=0.7777777777777778, fmeasure=0.7777777777777778), 'rouge2': Score(precision=0.625, recall=0.625, fmeasure=0.625), 'rougeL': Score(precision=0.7777777777777778, recall=0.7777777777777778, fmeasure=0.7777777777777778)}
WER Score: 0.2222222222222222
CER Score: 1.0


---
## 2. Modern GenAI Evaluation: LLM-as-a-Judge

While BLEU and ROUGE are fast, they suffer from severe limitations:
- A valid paraphrase with zero lexical overlap receives a score of 0.0!
- They cannot evaluate factual correctness, reasoning validity, tone, or safety.

**LLM-as-a-Judge** uses a strong judge model (e.g. GPT-4, Claude 3.5, or Gemini 1.5 Pro) with detailed evaluation rubrics.

### Two Primary Evaluation Modes:
1. **Pointwise Evaluation**: Score a single response on a Likert scale (1 to 5) across criteria (Relevance, Accuracy, Clarity).
2. **Pairwise Comparison**: Compare Response A vs. Response B head-to-head to compute win rates.

In [ ]:
# %pip install litellm pydantic pandas rich tabulate -q
import json
import os
from pydantic import BaseModel, Field
from typing import Literal, Optional
import pandas as pd

## 1. Defining Structured Schemas for Evaluation

In [ ]:
class PointwiseEvaluation(BaseModel):
    relevance_score: int = Field(description="Score from 1 (irrelevant) to 5 (perfectly answers the query)", ge=1, le=5)
    accuracy_score: int = Field(description="Score from 1 (factually incorrect) to 5 (completely accurate)", ge=1, le=5)
    clarity_score: int = Field(description="Score from 1 (confusing) to 5 (crystal clear, well-structured)", ge=1, le=5)
    overall_score: float = Field(description="Overall holistic score (1-5)")
    reasoning: str = Field(description="Detailed critique explaining why this score was assigned")

class PairwiseEvaluation(BaseModel):
    winner: Literal["model_a", "model_b", "tie"] = Field(description="Which model provided the superior response")
    model_a_critique: str = Field(description="Strengths and weaknesses of Model A")
    model_b_critique: str = Field(description="Strengths and weaknesses of Model B")
    rationale: str = Field(description="Justification for the chosen winner")

## 2. Pointwise Evaluation (Direct Scoring with Rubrics)

A well-engineered Judge prompt requires **explicit evaluation rubrics** to reduce judge variance and subjective drift.

In [ ]:
POINTWISE_JUDGE_PROMPT = """You are an expert impartial judge evaluating the quality of an AI assistant's response.

### Instruction to Evaluate:
{user_instruction}

### Assistant Response:
{assistant_response}

### Evaluation Criteria:
1. **Relevance (1-5)**: Does the response directly address what was asked without unnecessary fluff?
2. **Accuracy (1-5)**: Is the technical and conceptual information sound, factual, and correct?
3. **Clarity (1-5)**: Is the formatting clean, easy to parse, and well-explained?

### Output Format:
Respond in valid JSON matching this schema:
{
  "relevance_score": <int 1-5>,
  "accuracy_score": <int 1-5>,
  "clarity_score": <int 1-5>,
  "overall_score": <float 1.0-5.0>,
  "reasoning": "<detailed step-by-step critique>"
}
"""

def evaluate_pointwise(instruction: str, response: str, llm_fn=None):
    """
    Evaluates a single model output using the judge prompt.
    """
    prompt = POINTWISE_JUDGE_PROMPT.format(
        user_instruction=instruction,
        assistant_response=response
    )
    if llm_fn is None:
        # Simulated judge output for demonstration
        return PointwiseEvaluation(
            relevance_score=5,
            accuracy_score=4,
            clarity_score=5,
            overall_score=4.7,
            reasoning="The response directly explains transformer self-attention with correct mathematical matrix formulation and clean markdown structure."
        )
    return llm_fn(prompt)

## 3. Pairwise Comparison & Debiasing Position Bias

LLM judges are susceptible to **Position Bias** (the tendency to favor the first candidate `Model A` or second `Model B`).
To eliminate position bias, we execute a **two-pass swapped evaluation**:
- Pass 1: Present (Model A, Model B)
- Pass 2: Present (Model B, Model A)
- Final Winner: Declared only if consistent across both presentations, otherwise scored as a `tie`.

In [ ]:
PAIRWISE_JUDGE_PROMPT = """You are an expert judge comparing two AI assistants (Response A vs Response B).

### User Query:
{query}

### Response A:
{response_a}

### Response B:
{response_b}

### Instructions:
Compare both responses based on helpfulness, accuracy, clarity, and conciseness.
Return JSON with keys: winner ('model_a', 'model_b', or 'tie'), model_a_critique, model_b_critique, rationale.
"""

def evaluate_pairwise_debiased(query: str, response_a: str, response_b: str, llm_judge_fn=None):
    """
    Runs position-swapped pairwise evaluation to eliminate positional bias.
    """
    # Pass 1: (A, B)
    prompt_pass1 = PAIRWISE_JUDGE_PROMPT.format(query=query, response_a=response_a, response_b=response_b)
    # Pass 2: (B, A) - Swapped
    prompt_pass2 = PAIRWISE_JUDGE_PROMPT.format(query=query, response_a=response_b, response_b=response_a)
    
    # In real implementation, invoke llm_judge_fn(prompt_pass1) and llm_judge_fn(prompt_pass2)
    result1_winner = "model_a"   # Response A won in slot 1
    result2_winner = "model_b"   # In slot 2 (where Response A was in slot B), it won again
    
    # Verify consistency across swap
    if result1_winner == "model_a" and result2_winner == "model_b":
        consensus_winner = "Model A (True Winner)"
    elif result1_winner == "model_b" and result2_winner == "model_a":
        consensus_winner = "Model B (True Winner)"
    else:
        consensus_winner = "Inconclusive / Tie (Affected by Position Bias)"
        
    return consensus_winner

## 4. Key Biases in LLM Judges & Mitigation Strategies

| Bias Type | Description | Mitigation Strategy |
| :--- | :--- | :--- |
| **Position Bias** | Tendency to prefer Candidate 1 over Candidate 2. | Position swapping & bidirectional average. |
| **Verbosity Bias** | Giving higher scores to longer, wordy answers. | Explicit length normalization or conciseness penalty in rubric. |
| **Self-Enhancement Bias** | Models preferring outputs generated by their own family (e.g. GPT-4 preferring GPT-3.5). | Using multi-model judge ensembles (Claude 3.5 Sonnet + GPT-4o + Gemini 1.5 Pro). |
| **Authority / Formatting Bias** | High scores for pretty markdown tables even with false facts. | Reference-guided factual checks and automated unit-test validation. |

---
## 3. Standard Benchmarks: MMLU & `lm-evaluation-harness`

Overlap metrics and LLM judges score *open-ended* output. **Benchmarks** instead score a model on fixed datasets with known answers, so results are reproducible and comparable across models. The common ones:

| Benchmark | Measures | Format | Scoring |
|---|---|---|---|
| **MMLU** / MMLU-Pro | Broad knowledge, 57 subjects | Multiple choice | Accuracy |
| **HellaSwag**, **ARC**, **WinoGrande** | Commonsense / science reasoning | Multiple choice | Accuracy (`acc_norm`) |
| **TruthfulQA** | Resistance to common misconceptions | MC / generation | MC accuracy |
| **GSM8K**, **MATH** | Grade-school / competition math | Free-form, final answer | Exact match |
| **HumanEval**, **MBPP** | Code generation | Write a function | pass@k (run unit tests) |
| **IFEval** | Instruction following | Verifiable constraints | Rule-based pass rate |
| **MT-Bench**, **Chatbot Arena** | Chat quality | Open-ended | LLM judge / human Elo |

**How multiple-choice benchmarks are scored.** The harness does not ask the model to "write the answer". It formats the question, appends `Answer:`, and compares the **next-token log-probability** of ` A`, ` B`, ` C`, ` D`; the highest wins. Below we re-implement exactly this for MMLU on a small subset, so you can see what `lm-evaluation-harness` does under the hood.

In [1]:
import torch, time
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
MODEL = "Qwen/Qwen2.5-0.5B"
bench_tok = AutoTokenizer.from_pretrained(MODEL)
bench_model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device).eval()

LETTERS = "ABCD"
LETTER_IDS = [bench_tok.encode(" " + l)[0] for l in LETTERS]   # token ids for " A", " B", " C", " D"

def format_example(ex, with_answer):
    prompt = ex["question"].strip() + "\n"
    prompt += "".join(f"{l}. {c}\n" for l, c in zip(LETTERS, ex["choices"])) + "Answer:"
    return prompt + (f" {LETTERS[ex['answer']]}\n\n" if with_answer else "")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [2]:
@torch.no_grad()
def mmlu_accuracy(subject, n_test=50, n_shot=0):
    ds = load_dataset("cais/mmlu", subject)
    shots = "".join(format_example(e, True) for e in ds["dev"].select(range(n_shot)))
    test = ds["test"].select(range(min(n_test, len(ds["test"]))))
    correct = []
    for ex in test:
        x = bench_tok(shots + format_example(ex, False), return_tensors="pt").to(device)
        next_token_logits = bench_model(**x).logits[0, -1]
        pred = next_token_logits[LETTER_IDS].argmax().item()     # compare only A/B/C/D logits
        correct.append(pred == ex["answer"])
    return correct

SUBJECTS = ["abstract_algebra", "high_school_geography", "college_biology",
            "high_school_us_history", "computer_security", "philosophy"]
results = {}
t0 = time.time()
for s in SUBJECTS:
    results[s] = {"0-shot": mmlu_accuracy(s, 50, 0), "5-shot": mmlu_accuracy(s, 50, 5)}
print(f"done in {time.time()-t0:.0f}s on {device}")

college_biology/test-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B / 31.8kB            

college_biology/test-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

college_biology/validation-00000-of-0000(…): reconstructing file:   0%|          |  0.00B / 6.90kB            

college_biology/validation-00000-of-0000(…): downloading bytes:           |  0.00B            

college_biology/dev-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 4.27kB            

college_biology/dev-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

... (27 outputs left out)

Generating test split:   0%|          | 0/311 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/34 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

done in 92s on mps


In [3]:
import pandas as pd, numpy as np

def summarize(flags):
    p = np.mean(flags); se = np.sqrt(p * (1 - p) / len(flags))
    return p, se

rows = []
for s, r in results.items():
    row = {"subject": s}
    for k, flags in r.items():
        p, se = summarize(flags); row[k] = f"{p:.2f} ± {se:.2f}"
    rows.append(row)
overall = {k: summarize(np.concatenate([results[s][k] for s in SUBJECTS])) for k in ["0-shot", "5-shot"]}
rows.append({"subject": "ALL (pooled questions)", **{k: f"{p:.2f} ± {se:.2f}" for k, (p, se) in overall.items()}})
pd.DataFrame(rows).set_index("subject")

,0-shot,5-shot
subject,,
abstract_algebra,0.44 ± 0.07,0.42 ± 0.07
high_school_geography,0.56 ± 0.07,0.50 ± 0.07
college_biology,0.54 ± 0.07,0.48 ± 0.07
high_school_us_history,0.52 ± 0.07,0.46 ± 0.07
computer_security,0.70 ± 0.06,0.74 ± 0.06
philosophy,0.52 ± 0.07,0.58 ± 0.07
ALL (pooled questions),0.55 ± 0.03,0.53 ± 0.03


**Reading the numbers**
- **Random guessing = 0.25.** A 0.5B base model should land clearly above that, but far below frontier models (>0.85 on MMLU).
- **The ± is the standard error** $\sqrt{p(1-p)/n}$. With only 50 questions per subject it is ~±0.07, so most per-subject differences, and the 0-shot vs 5-shot gap, are **not statistically meaningful**. Real MMLU runs use all ~14k questions. Always report uncertainty before ranking models.
- **Few-shot can hurt small models**: extra examples lengthen the prompt and can shift the answer-letter prior. Prompt format changes scores by several points, which is why leaderboards fix the format.

### Running the real harness

The code above is a teaching version. For reproducible, comparable numbers use [EleutherAI `lm-evaluation-harness`](https://github.com/EleutherAI/lm-evaluation-harness), which is also what the Hugging Face Open LLM Leaderboard used:

```bash
pip install lm-eval
lm_eval --model hf --model_args pretrained=Qwen/Qwen2.5-0.5B \
        --tasks mmlu,hellaswag,arc_easy,gsm8k --num_fewshot 5 --batch_size 8 \
        --limit 100 --output_path results/        # --limit for a quick smoke test
```

You can point it at your own LoRA adapter (`--model_args pretrained=<base model>,peft=<adapter dir>`, for example the BabyQwen-3 adapter on top of `baby_qwen_sft`) to check whether fine-tuning caused **catastrophic forgetting** on general knowledge.

### Pitfalls of benchmark-based evaluation

| Pitfall | Why it matters |
|---|---|
| **Contamination** | Test questions leak into web-scale pretraining data (see the dedup notebook 4.2), inflating scores |
| **Saturation** | Models approach the ceiling (MMLU), so it stops discriminating; harder successors like MMLU-Pro and GPQA appear |
| **Format sensitivity** | Prompt template, shot count and answer extraction move scores by several points |
| **Goodhart's law** | Optimizing for a benchmark makes it a worse measure of real quality |
| **Task mismatch** | Multiple-choice accuracy says little about open-ended chat, which is why we also use LLM judges (Section 2) and human Elo (Chatbot Arena) |

---
## 4. Summary & Modern Evaluation Taxonomy

In [ ]:
eval_taxonomy = [
    {
        "Tier": "Tier 1: Lexical Overlap",
        "Metrics": "BLEU, ROUGE, METEOR",
        "Cost": "Free (microseconds)",
        "Limitation": "Penalizes valid paraphrases; blind to factual errors"
    },
    {
        "Tier": "Tier 2: Embedding Similarity",
        "Metrics": "BERTScore, MoverScore",
        "Cost": "Very Low (GPU embeddings)",
        "Limitation": "Captures semantic similarity but misses logical reasoning"
    },
    {
        "Tier": "Tier 3: LLM-as-a-Judge",
        "Metrics": "Pointwise rubrics, Pairwise win rates (MT-Bench, AlpacaEval)",
        "Cost": "Moderate (LLM API tokens)",
        "Limitation": "Susceptible to position, verbosity, and self-enhancement biases"
    },
    {
        "Tier": "Tier 4: Execution / Verifiable",
        "Metrics": "Unit tests (HumanEval), Math solvers (GSM8K/MATH)",
        "Cost": "Low (Docker execution)",
        "Limitation": "Only applicable to tasks with objective ground-truth verification"
    }
]

df_taxonomy = pd.DataFrame(eval_taxonomy)
print("=== Comprehensive LLM Evaluation Taxonomy ===")
print(df_taxonomy.to_string(index=False))

## 5. Summary & Key Takeaways

1. **Multi-Faceted Evaluation**: Never rely on a single metric. Combine n-gram metrics (for fluency check) with LLM-as-a-Judge (for depth and relevance) and execution benchmarks (for code/math).
2. **Debiasing LLM Judges**: Always swap the candidate positions ($A \to B$ and $B \to A$) when performing pairwise comparisons to eliminate inherent position bias.
3. **Structured Outputs**: Use Pydantic or constrained JSON schemas when querying LLM judges to ensure programmatic parsing and auditability.
6. **Benchmarks complement judges**: use standard benchmarks (MMLU, GSM8K, HumanEval via `lm-evaluation-harness`) for reproducible capability and regression checks, and always report uncertainty and watch for contamination.